# 🎰 Multi-Armed Bandit Playground



## 📋 Table of Contents

1. [Introduction & Problem Framing](#1-introduction)
2. [Mathematical Foundations](#2-math)
3. [Environment Design](#3-environment)
4. [Algorithm Implementations](#4-algorithms)
5. [Experiment Framework](#5-framework)
6. [Visualization Suite](#6-visualizations)
7. [Core Experiments & Analysis](#7-experiments)
8. [Advanced Experiments](#8-advanced)
9. [Business Strategy Memo](#9-business)
10. [Interactive Explorer](#10-interactive)
11. [Utilities & Export](#11-utilities)
12. [Conclusions & Future Work](#12-conclusions)

---
## 1. Introduction & Problem Framing <a id="1-introduction"></a>

### The Problem

Imagine you're a product manager testing **5 different homepage designs**. Each design converts visitors at an unknown rate. You need to figure out which design is best — but every visitor shown a bad design is **lost revenue**.

This is the **multi-armed bandit problem**: balancing **exploration** (trying different options to learn) with **exploitation** (using what you already know to maximize reward).

### Why Not A/B Testing?

Traditional A/B tests split traffic equally across all variants for a fixed period, then pick the winner. This works, but:

- **Wastes traffic** on clearly inferior variants
- **Fixed duration** — can't adapt early
- **Binary outcome** — no gradual learning

Bandit algorithms **adapt in real-time**, shifting traffic toward better-performing variants while still exploring.

### What We'll Build

In this notebook, we simulate product experimentation scenarios and compare five bandit algorithms on:

- **Cumulative reward** — total conversions captured
- **Cumulative regret** — reward lost vs. always picking the best arm
- **Convergence speed** — how fast each algorithm finds the best option
- **Robustness** — performance under low traffic, many options, and changing environments

---
## 2. Mathematical Foundations <a id="2-math"></a>

### Regret

At each time step $t$, we choose arm $a_t$ and receive reward $r_t$. The **optimal arm** $a^*$ has the highest expected reward $\mu^*$. **Regret** measures the cost of not always picking the best:

$$R_T = \sum_{t=1}^{T} (\mu^* - \mu_{a_t})$$

Good algorithms achieve **sublinear regret** — regret grows slower than $T$, meaning we converge to the optimal arm.

### Key Algorithm Ideas

| Algorithm | Core Idea |
|---|---|
| **Epsilon-Greedy** | Exploit the best-known arm $(1-\varepsilon)$ of the time; explore randomly $\varepsilon$ of the time |
| **Optimistic Initial Values** | Initialize expected rewards high, so all arms get tried early as reality disappoints |
| **UCB1** | Pick the arm with highest $\bar{x}_a + c\sqrt{\frac{\ln t}{n_a}}$ — balances mean reward with uncertainty |
| **Thompson Sampling** | Sample from each arm's posterior $\text{Beta}(\alpha_a, \beta_a)$, pick the highest sample |
| **Sliding-Window TS** | Same as Thompson, but only uses the last $W$ observations — adapts to changing rewards |

In [ ]:
# ============================================================
# Imports & Configuration
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from collections import defaultdict
import warnings
import os

# Reproducibility
GLOBAL_SEED = 42

# Plot styling
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 100,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

# Color palette for algorithms
ALGO_COLORS = {
    'Epsilon-Greedy': '#E74C3C',
    'Optimistic Init': '#3498DB',
    'UCB1': '#2ECC71',
    'Thompson Sampling': '#9B59B6',
    'Sliding-Window TS': '#F39C12',
}

# Output directory for saved plots
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Setup complete. NumPy version:', np.__version__)

---
## 3. Environment Design <a id="3-environment"></a>

The `BanditEnvironment` simulates a product experiment with configurable arms (variants), reward probabilities, and optional non-stationary drift.

**Key features:**
- Configurable number of arms and reward probabilities
- Bernoulli reward distribution (click/no-click)
- Stationary or non-stationary (drifting) rewards
- Reproducible via random seeds

In [ ]:
# ============================================================
# BanditEnvironment
# ============================================================

class BanditEnvironment:
    """Simulates a multi-armed bandit product experiment.
    
    Parameters
    ----------
    reward_probs : list of float
        True reward probability for each arm (Bernoulli).
    stationary : bool
        If False, reward probabilities drift over time.
    drift_rate : float
        Std dev of Gaussian noise added to probs each step (non-stationary only).
    seed : int or None
        Random seed for reproducibility.
    """
    
    def __init__(self, reward_probs, stationary=True, drift_rate=0.001, seed=None):
        self.initial_probs = np.array(reward_probs, dtype=float)
        self.reward_probs = self.initial_probs.copy()
        self.n_arms = len(reward_probs)
        self.stationary = stationary
        self.drift_rate = drift_rate
        self.rng = np.random.RandomState(seed)
        self.step_count = 0
    
    def pull(self, arm):
        """Pull an arm and return a Bernoulli reward (0 or 1)."""
        reward = self.rng.random() < self.reward_probs[arm]
        self.step_count += 1
        if not self.stationary:
            self._drift()
        return float(reward)
    
    def _drift(self):
        """Apply bounded random walk to reward probabilities."""
        noise = self.rng.normal(0, self.drift_rate, self.n_arms)
        self.reward_probs = np.clip(self.reward_probs + noise, 0.01, 0.99)
    
    def get_optimal_arm(self):
        """Return the index of the current best arm."""
        return int(np.argmax(self.reward_probs))
    
    def get_optimal_reward(self):
        """Return the reward probability of the current best arm."""
        return float(np.max(self.reward_probs))
    
    def reset(self):
        """Reset environment to initial state."""
        self.reward_probs = self.initial_probs.copy()
        self.step_count = 0
    
    def __repr__(self):
        status = 'stationary' if self.stationary else f'non-stationary (drift={self.drift_rate})'
        return f'BanditEnvironment(arms={self.n_arms}, {status}, probs={self.reward_probs.round(3)})'


# Quick validation 
env = BanditEnvironment(reward_probs=[0.1, 0.3, 0.5, 0.7, 0.9], seed=42)
print(env)
print(f'Optimal arm: {env.get_optimal_arm()} (p={env.get_optimal_reward():.1f})')
print(f'Sample pulls from arm 4: {[env.pull(4) for _ in range(10)]}')

# Non-stationary test
env_ns = BanditEnvironment(reward_probs=[0.3, 0.5, 0.7], stationary=False, drift_rate=0.01, seed=42)
for _ in range(1000):
    env_ns.pull(0)
print(f'\nAfter 1000 steps with drift: {env_ns}')
print(f'Probs stayed in [0.01, 0.99]: {all(0.01 <= p <= 0.99 for p in env_ns.reward_probs)}')

---
## 4. Algorithm Implementations <a id="4-algorithms"></a>

All algorithms share a common `BanditAlgorithm` interface and track cumulative reward, cumulative regret, and arm selection counts.

| Class | Key Parameter | Strategy |
|---|---|---|
| `EpsilonGreedy` | `epsilon` | Random explore at rate ε |
| `OptimisticInitialValues` | `initial_value` | Greedy on inflated Q-values |
| `UCB1` | `c` | Upper confidence bound |
| `ThompsonSampling` | — | Beta posterior sampling |
| `SlidingWindowTS` | `window_size` | Windowed Beta posterior |

In [ ]:
# ============================================================
# Bandit Algorithms
# ============================================================

class BanditAlgorithm:
    """Base class for all bandit algorithms.
    
    Subclasses must implement select_arm() and update().
    """
    
    def __init__(self, n_arms, name='Base'):
        self.n_arms = n_arms
        self.name = name
        self.reset()
    
    def select_arm(self):
        raise NotImplementedError
    
    def update(self, arm, reward, optimal_reward=0.0):
        """Update state after observing a reward."""
        self.arm_counts[arm] += 1
        self.total_reward += reward
        self.total_regret += (optimal_reward - reward)
        self.reward_history.append(self.total_reward)
        self.regret_history.append(self.total_regret)
    
    def reset(self):
        """Reset all tracking state."""
        self.arm_counts = np.zeros(self.n_arms)
        self.total_reward = 0.0
        self.total_regret = 0.0
        self.reward_history = []
        self.regret_history = []
    
    def __repr__(self):
        return f'{self.name}(arms={self.n_arms})'


# 1. Epsilon-Greedy

class EpsilonGreedy(BanditAlgorithm):
    """Explores randomly with probability epsilon, exploits otherwise."""
    
    def __init__(self, n_arms, epsilon=0.1, seed=None):
        self.epsilon = epsilon
        self.q_values = np.zeros(n_arms)
        self.rng = np.random.RandomState(seed)
        super().__init__(n_arms, name='Epsilon-Greedy')
    
    def select_arm(self):
        if self.rng.random() < self.epsilon:
            return self.rng.randint(self.n_arms)
        return int(np.argmax(self.q_values))
    
    def update(self, arm, reward, optimal_reward=0.0):
        super().update(arm, reward, optimal_reward)
        # Incremental mean update
        n = self.arm_counts[arm]
        self.q_values[arm] += (reward - self.q_values[arm]) / n
    
    def reset(self):
        super().reset()
        self.q_values = np.zeros(self.n_arms)


# 2. Optimistic Initial Values

class OptimisticInitialValues(BanditAlgorithm):
    """Initializes Q-values high to encourage early exploration."""
    
    def __init__(self, n_arms, initial_value=5.0, seed=None):
        self.initial_value = initial_value
        self.q_values = np.full(n_arms, initial_value, dtype=float)
        self.rng = np.random.RandomState(seed)
        super().__init__(n_arms, name='Optimistic Init')
    
    def select_arm(self):
        # Break ties randomly
        max_q = np.max(self.q_values)
        candidates = np.where(self.q_values == max_q)[0]
        return int(self.rng.choice(candidates))
    
    def update(self, arm, reward, optimal_reward=0.0):
        super().update(arm, reward, optimal_reward)
        n = self.arm_counts[arm]
        self.q_values[arm] += (reward - self.q_values[arm]) / n
    
    def reset(self):
        super().reset()
        self.q_values = np.full(self.n_arms, self.initial_value, dtype=float)


# 3. UCB1 

class UCB1(BanditAlgorithm):
    """Upper Confidence Bound — balances exploitation with uncertainty bonus."""
    
    def __init__(self, n_arms, c=2.0, seed=None):
        self.c = c
        self.q_values = np.zeros(n_arms)
        self.rng = np.random.RandomState(seed)
        super().__init__(n_arms, name='UCB1')
    
    def select_arm(self):
        total_counts = np.sum(self.arm_counts)
        # Play each arm once first
        for arm in range(self.n_arms):
            if self.arm_counts[arm] == 0:
                return arm
        ucb_values = self.q_values + self.c * np.sqrt(np.log(total_counts) / self.arm_counts)
        return int(np.argmax(ucb_values))
    
    def update(self, arm, reward, optimal_reward=0.0):
        super().update(arm, reward, optimal_reward)
        n = self.arm_counts[arm]
        self.q_values[arm] += (reward - self.q_values[arm]) / n
    
    def reset(self):
        super().reset()
        self.q_values = np.zeros(self.n_arms)


# 4. Thompson Sampling (Beta-Bernoulli)

class ThompsonSampling(BanditAlgorithm):
    """Bayesian approach — samples from Beta posterior for each arm."""
    
    def __init__(self, n_arms, seed=None):
        self.alpha = np.ones(n_arms)  # successes + 1
        self.beta_param = np.ones(n_arms)   # failures + 1
        self.rng = np.random.RandomState(seed)
        super().__init__(n_arms, name='Thompson Sampling')
    
    def select_arm(self):
        samples = self.rng.beta(self.alpha, self.beta_param)
        return int(np.argmax(samples))
    
    def update(self, arm, reward, optimal_reward=0.0):
        super().update(arm, reward, optimal_reward)
        self.alpha[arm] += reward
        self.beta_param[arm] += (1 - reward)
    
    def reset(self):
        super().reset()
        self.alpha = np.ones(self.n_arms)
        self.beta_param = np.ones(self.n_arms)


# 5. Sliding-Window Thompson Sampling

class SlidingWindowTS(BanditAlgorithm):
    """Thompson Sampling using only the last W observations per arm.
    
    Adapts to non-stationary environments by forgetting old data.
    """
    
    def __init__(self, n_arms, window_size=200, seed=None):
        self.window_size = window_size
        self.windows = [[] for _ in range(n_arms)]
        self.rng = np.random.RandomState(seed)
        super().__init__(n_arms, name='Sliding-Window TS')
    
    def select_arm(self):
        samples = np.zeros(self.n_arms)
        for arm in range(self.n_arms):
            w = self.windows[arm][-self.window_size:] if self.windows[arm] else []
            alpha = sum(w) + 1
            beta_param = len(w) - sum(w) + 1
            samples[arm] = self.rng.beta(alpha, beta_param)
        return int(np.argmax(samples))
    
    def update(self, arm, reward, optimal_reward=0.0):
        super().update(arm, reward, optimal_reward)
        self.windows[arm].append(reward)
        # Keep only recent observations to save memory
        if len(self.windows[arm]) > self.window_size * 2:
            self.windows[arm] = self.windows[arm][-self.window_size:]
    
    def reset(self):
        super().reset()
        self.windows = [[] for _ in range(self.n_arms)]


# --- Quick validation ---
env_test = BanditEnvironment(reward_probs=[0.2, 0.5, 0.8], seed=0)
algos = [
    EpsilonGreedy(3, epsilon=0.1, seed=0),
    OptimisticInitialValues(3, initial_value=5.0, seed=0),
    UCB1(3, c=2.0, seed=0),
    ThompsonSampling(3, seed=0),
    SlidingWindowTS(3, window_size=200, seed=0),
]

for algo in algos:
    for _ in range(100):
        arm = algo.select_arm()
        reward = env_test.pull(arm)
        algo.update(arm, reward, env_test.get_optimal_reward())
    env_test.reset()
    print(f'{algo.name:20s} | reward={algo.total_reward:.0f} | regret={algo.total_regret:.1f} | counts={algo.arm_counts}')

---
## 5. Experiment Framework <a id="5-framework"></a>

*Coming in Step 5*

In [ ]:
# ExperimentRunner

---
## 6. Visualization Suite <a id="6-visualizations"></a>

*Coming in Step 6*

In [ ]:
# Visualization functions

---
## 7. Core Experiments & Analysis <a id="7-experiments"></a>

*Coming in Step 7*

In [ ]:
# Core experiments

---
## 8. Advanced Experiments <a id="8-advanced"></a>

*Coming in Step 8*

In [ ]:
# Advanced experiments

---
## 9. Business Strategy Memo <a id="9-business"></a>

*TBD

---
## 10. Interactive Explorer <a id="10-interactive"></a>

*Coming in Step 10*

In [ ]:
# Interactive widgets

---
## 11. Utilities & Export <a id="11-utilities"></a>

*Coming in Step 10*

In [ ]:
# Export utilities

---
## 12. Conclusions & Future Work <a id="12-conclusions"></a>
